# Messy Mashup - Variant 2: Ensemble (Target: 85+)
Trains CNN + XGBoost + AST, uses **AST + CNN weighted ensemble** with TTA.

**For submission:** Set `INFERENCE_ONLY = True` and point `MODEL_PATH` to your uploaded weights dataset.

In [ ]:
!pip install librosa transformers wandb xgboost joblib -q

In [ ]:
import os, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from tqdm import tqdm
from transformers import ASTForAudioClassification, ASTFeatureExtractor
import matplotlib.pyplot as plt
from collections import Counter
import joblib

In [ ]:
# ============================================================
# TOGGLE: Set True when submitting (skips training, loads weights)
# ============================================================
INFERENCE_ONLY = False
MODEL_PATH = '/kaggle/input/messy-mashup-models'  # path to your uploaded weights dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000
SAMPLES = SR * 5
GENRES = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2ID = {g:i for i,g in enumerate(GENRES)}
ID2GENRE = {i:g for g,i in GENRE2ID.items()}

root = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
mashup_dir = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/'
print(f'Device: {DEVICE}')
print(f'Mode: {"INFERENCE ONLY" if INFERENCE_ONLY else "TRAINING + INFERENCE"}')

In [ ]:
# W&B setup (only needed during training)
if not INFERENCE_ONLY:
    import wandb
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret("WANDB"))

## EDA

In [ ]:
if not INFERENCE_ONLY:
    genre_counts = Counter()
    for g in GENRES:
        genre_counts[g] = len(os.listdir(os.path.join(root, g)))

    plt.figure(figsize=(10, 4))
    plt.bar(genre_counts.keys(), genre_counts.values(), color='steelblue')
    plt.xticks(rotation=45); plt.title('Genre Distribution')
    plt.tight_layout(); plt.show()
    print(genre_counts)

In [ ]:
if not INFERENCE_ONLY:
    for g in GENRES[:3]:
        gpath = os.path.join(root, g)
        song = os.listdir(gpath)[0]
        for stem in ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']:
            fpath = os.path.join(gpath, song, stem)
            if os.path.exists(fpath):
                y, sr_ = librosa.load(fpath, sr=None)
                print(f'  {g}/{song}/{stem}: {len(y)/sr_:.2f}s')

## Audio Utilities

In [ ]:
def load_audio(path):
    y, _ = librosa.load(path, sr=SR)
    if len(y) > SAMPLES:
        start = np.random.randint(0, len(y) - SAMPLES)
        y = y[start:start + SAMPLES]
    else:
        y = np.pad(y, (0, max(0, SAMPLES - len(y))))
    return y

def to_mel(y):
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
    mel = librosa.power_to_db(mel)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)

def augment_audio(y):
    y = y * np.random.uniform(0.7, 1.3)
    shift = np.random.randint(-SR // 2, SR // 2)
    y = np.roll(y, shift)
    if random.random() < 0.3:
        y = y + np.random.randn(len(y)) * 0.005
    return y

STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

## Model Definitions

In [ ]:
class StrongCNN(nn.Module):
    """4-block CNN with BatchNorm and Dropout."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.3),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))

feature_extractor = ASTFeatureExtractor.from_pretrained(
    'MIT/ast-finetuned-audioset-10-10-0.4593'
)

## Datasets

In [ ]:
if not INFERENCE_ONLY:
    class CNNDataset(Dataset):
        def __init__(self, root, augment=False):
            self.samples = []
            self.augment = augment
            for g in GENRES:
                gpath = os.path.join(root, g)
                for song in os.listdir(gpath):
                    self.samples.append((os.path.join(gpath, song), GENRE2ID[g]))

        def __len__(self): return len(self.samples)

        def __getitem__(self, idx):
            path, label = self.samples[idx]
            stems = list(STEMS)
            if self.augment and random.random() < 0.5 and len(stems) > 1:
                stems.remove(random.choice(stems))
            y = sum([load_audio(os.path.join(path, f)) for f in stems])
            if self.augment:
                y = augment_audio(y)
            mel = torch.tensor(to_mel(y)).unsqueeze(0)
            return mel, label

    class ASTTrainDataset(Dataset):
        def __init__(self, root):
            self.samples = []
            for g in GENRES:
                gpath = os.path.join(root, g)
                for song in os.listdir(gpath):
                    self.samples.append((os.path.join(gpath, song), GENRE2ID[g]))

        def __len__(self): return len(self.samples)

        def __getitem__(self, idx):
            path, label = self.samples[idx]
            stems = list(STEMS)
            if random.random() < 0.5 and len(stems) > 1:
                stems.remove(random.choice(stems))
            y = sum([load_audio(os.path.join(path, f)) for f in stems])
            y = augment_audio(y)
            inputs = feature_extractor(y, sampling_rate=SR, return_tensors='pt')
            return inputs['input_values'].squeeze(0), label

    cnn_full_dataset = CNNDataset(root, augment=True)
    cnn_noaug_dataset = CNNDataset(root, augment=False)

    indices = list(range(len(cnn_noaug_dataset)))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

    cnn_train_loader = DataLoader(torch.utils.data.Subset(cnn_full_dataset, train_idx),
                                  batch_size=16, shuffle=True)
    cnn_val_loader = DataLoader(torch.utils.data.Subset(cnn_noaug_dataset, val_idx),
                                batch_size=16, shuffle=False)

    ast_train_loader = DataLoader(ASTTrainDataset(root), batch_size=8, shuffle=True)

    print(f'CNN train: {len(train_idx)}, val: {len(val_idx)}')
    print(f'AST train: {len(ast_train_loader.dataset)}')

---
## Model 1: Strong CNN (From Scratch)

In [ ]:
if not INFERENCE_ONLY:
    wandb.init(project='24f1001895-t12026', name='CNN-strong-v2')

    cnn_model = StrongCNN().to(DEVICE)
    cnn_opt = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
    cnn_scheduler = CosineAnnealingLR(cnn_opt, T_max=20)

    best_cnn_f1 = 0
    for epoch in range(20):
        cnn_model.train()
        for x, y in cnn_train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            cnn_opt.zero_grad()
            loss = F.cross_entropy(cnn_model(x), y, label_smoothing=0.1)
            loss.backward()
            cnn_opt.step()
        cnn_scheduler.step()

        cnn_model.eval()
        preds, true = [], []
        with torch.no_grad():
            for x, y in cnn_val_loader:
                x = x.to(DEVICE)
                pred = torch.argmax(cnn_model(x), 1).cpu().numpy()
                preds.extend(pred)
                true.extend(y.numpy())

        cnn_f1 = f1_score(true, preds, average='macro')
        if cnn_f1 > best_cnn_f1:
            best_cnn_f1 = cnn_f1
            torch.save(cnn_model.state_dict(), 'best_cnn.pth')

        if (epoch + 1) % 5 == 0:
            print(f'CNN Epoch {epoch+1} | Val F1: {cnn_f1:.4f} | Best: {best_cnn_f1:.4f}')
        wandb.log({'cnn_val_f1': cnn_f1, 'epoch': epoch+1})

    cnn_model.load_state_dict(torch.load('best_cnn.pth'))
    print(f'CNN Best Val F1: {best_cnn_f1:.4f} | Saved best_cnn.pth')
    wandb.finish()

---
## Model 2: XGBoost (Classical ML)

In [ ]:
if not INFERENCE_ONLY:
    wandb.init(project='24f1001895-t12026', name='XGBoost-v2')

    cnn_noaug_loader_full = DataLoader(cnn_noaug_dataset, batch_size=16, shuffle=False)
    all_X, all_y = [], []
    for x, y in cnn_noaug_loader_full:
        for i in range(len(x)):
            all_X.append(x[i].numpy().flatten())
            all_y.append(y[i].item())

    X_tr = [all_X[i] for i in train_idx]
    y_tr = [all_y[i] for i in train_idx]
    X_vl = [all_X[i] for i in val_idx]
    y_vl = [all_y[i] for i in val_idx]

    xgb_model = XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', random_state=42)
    xgb_model.fit(X_tr, y_tr)

    xgb_train_f1 = f1_score(y_tr, xgb_model.predict(X_tr), average='macro')
    xgb_val_f1 = f1_score(y_vl, xgb_model.predict(X_vl), average='macro')

    print(f'XGBoost Train F1: {xgb_train_f1:.4f}')
    print(f'XGBoost Val F1:   {xgb_val_f1:.4f}')
    wandb.log({'xgb_train_f1': xgb_train_f1, 'xgb_val_f1': xgb_val_f1})

    joblib.dump(xgb_model, 'xgb_model.pkl')
    print('Saved xgb_model.pkl')
    wandb.finish()

---
## Model 3: AST (Pretrained) - 12 epochs + label smoothing

In [ ]:
if not INFERENCE_ONLY:
    wandb.init(project='24f1001895-t12026', name='AST-v2-ensemble')

    ast_model = ASTForAudioClassification.from_pretrained(
        'MIT/ast-finetuned-audioset-10-10-0.4593',
        num_labels=10, ignore_mismatched_sizes=True
    ).to(DEVICE)

    EPOCHS = 12
    ACCUM_STEPS = 2

    ast_opt = torch.optim.AdamW(ast_model.parameters(), lr=1e-5, weight_decay=0.01)
    scheduler = CosineAnnealingLR(ast_opt, T_max=EPOCHS)

    for param in ast_model.audio_spectrogram_transformer.parameters():
        param.requires_grad = False

    for epoch in range(EPOCHS):
        if epoch == 2:
            for param in ast_model.audio_spectrogram_transformer.parameters():
                param.requires_grad = True
            print('Unfroze encoder')

        ast_model.train()
        total_loss = 0
        ast_opt.zero_grad()

        for i, (x, y) in enumerate(tqdm(ast_train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')):
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = F.cross_entropy(ast_model(x).logits, y, label_smoothing=0.1) / ACCUM_STEPS
            loss.backward()
            total_loss += loss.item() * ACCUM_STEPS

            if (i + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(ast_model.parameters(), max_norm=1.0)
                ast_opt.step()
                ast_opt.zero_grad()

        if (i + 1) % ACCUM_STEPS != 0:
            torch.nn.utils.clip_grad_norm_(ast_model.parameters(), max_norm=1.0)
            ast_opt.step()
            ast_opt.zero_grad()

        scheduler.step()
        avg_loss = total_loss / len(ast_train_loader)
        lr = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch+1} | Loss: {avg_loss:.4f} | LR: {lr:.2e}')
        wandb.log({'ast_train_loss': avg_loss, 'ast_lr': lr, 'epoch': epoch+1})

    torch.save(ast_model.state_dict(), 'ast_best.pth')
    print('Saved ast_best.pth')
    wandb.finish()

---
## Model Comparison

In [ ]:
if not INFERENCE_ONLY:
    # Evaluate AST on training set for comparison
    ast_model.eval()
    ast_preds, ast_true = [], []
    with torch.no_grad():
        for x, y in DataLoader(ASTTrainDataset(root), batch_size=8, shuffle=False):
            x = x.to(DEVICE)
            pred = torch.argmax(ast_model(x).logits, 1).cpu().numpy()
            ast_preds.extend(pred)
            ast_true.extend(y.numpy())
    ast_f1 = f1_score(ast_true, ast_preds, average='macro')

    print('===== MODEL COMPARISON =====')
    print(f'CNN Val F1:     {best_cnn_f1:.4f}')
    print(f'XGBoost Val F1: {xgb_val_f1:.4f}')
    print(f'AST Train F1:   {ast_f1:.4f}')

    # W&B comparison run with table, bar chart, and summary metrics
    run = wandb.init(project='24f1001895-t12026', name='model-comparison-v2')

    # Summary metrics for cross-run overview
    wandb.summary['cnn_val_f1']  = round(best_cnn_f1, 4)
    wandb.summary['xgb_val_f1']  = round(xgb_val_f1, 4)
    wandb.summary['ast_f1']      = round(ast_f1, 4)
    wandb.summary['best_model']  = 'Ensemble (AST+CNN)'
    wandb.summary['best_f1']     = round(ast_f1, 4)  # AST dominates at 0.75 weight

    # Comparison table
    comparison_table = wandb.Table(
        columns=['Model', 'Type', 'Val_F1', 'In_Ensemble'],
        data=[
            ['StrongCNN', 'From Scratch', round(best_cnn_f1, 4), True],
            ['XGBoost',   'Classical ML', round(xgb_val_f1, 4), False],
            ['AST',       'Pretrained',   round(ast_f1, 4),      True],
        ]
    )
    wandb.log({'model_comparison_table': comparison_table})

    # Bar chart
    wandb.log({'model_comparison_bar': wandb.plot.bar(
        comparison_table, label='Model', value='Val_F1', title='Macro F1 by Model (Variant 2)'
    )})

    # Scalar metrics for cross-run comparison
    wandb.log({
        'cnn_val_f1':  round(best_cnn_f1, 4),
        'xgb_val_f1':  round(xgb_val_f1, 4),
        'ast_val_f1':  round(ast_f1, 4),
    })

    wandb.finish()

    # Local bar chart
    models = {'StrongCNN': best_cnn_f1, 'XGBoost': xgb_val_f1, 'AST': ast_f1}
    plt.figure(figsize=(8, 4))
    plt.barh(list(models.keys()), list(models.values()),
             color=['#3498db', '#2ecc71', '#9b59b6'])
    plt.xlabel('Macro F1')
    plt.title('Model Comparison — Variant 2 (Ensemble)')
    plt.xlim(0, 1)
    for i, v in enumerate(models.values()):
        plt.text(v + 0.01, i, f'{v:.4f}', va='center')
    plt.tight_layout()
    plt.show()

---
## Load Saved Models (for inference-only mode)

In [ ]:
if INFERENCE_ONLY:
    # Load CNN
    cnn_model = StrongCNN().to(DEVICE)
    cnn_model.load_state_dict(torch.load(f'{MODEL_PATH}/best_cnn.pth', map_location=DEVICE))
    cnn_model.eval()
    print('Loaded CNN from saved weights')

    # Load AST
    ast_model = ASTForAudioClassification.from_pretrained(
        'MIT/ast-finetuned-audioset-10-10-0.4593',
        num_labels=10, ignore_mismatched_sizes=True
    ).to(DEVICE)
    ast_model.load_state_dict(torch.load(f'{MODEL_PATH}/ast_best.pth', map_location=DEVICE))
    ast_model.eval()
    print('Loaded AST from saved weights')

    # XGBoost not used in ensemble but load if needed
    # xgb_model = joblib.load(f'{MODEL_PATH}/xgb_model.pkl')

---
## Inference: AST + CNN Ensemble with 7-chunk TTA

In [ ]:
def get_chunks(y, n=7):
    if len(y) <= SAMPLES:
        return [np.pad(y, (0, SAMPLES - len(y)))]
    step = (len(y) - SAMPLES) / max(n - 1, 1)
    chunks = []
    for i in range(n):
        start = int(i * step)
        chunks.append(y[start:start + SAMPLES])
    return chunks

def predict_ast_probs(y_audio):
    chunks = get_chunks(y_audio, 7)
    probs = []
    for c in chunks:
        inputs = feature_extractor(c, sampling_rate=SR, return_tensors='pt')
        with torch.no_grad():
            out = ast_model(inputs['input_values'].to(DEVICE)).logits
            probs.append(torch.softmax(out, 1).cpu().numpy()[0])
    return np.mean(probs, axis=0)

def predict_cnn_probs(y_audio):
    chunks = get_chunks(y_audio, 7)
    probs = []
    for c in chunks:
        mel = torch.tensor(to_mel(c)).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            out = cnn_model(mel)
            probs.append(torch.softmax(out, 1).cpu().numpy()[0])
    return np.mean(probs, axis=0)

def ensemble_predict(path):
    y, _ = librosa.load(path, sr=SR)
    ast_probs = predict_ast_probs(y)
    cnn_probs = predict_cnn_probs(y)
    final = 0.75 * ast_probs + 0.25 * cnn_probs
    return np.argmax(final)

In [ ]:
test_df = pd.read_csv('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv')

ast_model.eval()
cnn_model.eval()

preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Ensemble predicting'):
    path = os.path.join(mashup_dir, row['filename'])
    preds.append(ID2GENRE[ensemble_predict(path)])

submission = pd.DataFrame({'id': test_df['id'], 'genre': preds})
submission.to_csv('submission.csv', index=False)

print(f'Predictions: {len(submission)}')
print(submission['genre'].value_counts())
print(submission.head(10))

---
## Save Artifacts Summary

In [ ]:
if not INFERENCE_ONLY:
    print('='*50)
    print('SAVED ARTIFACTS:')
    print('  - best_cnn.pth')
    print('  - xgb_model.pkl')
    print('  - ast_best.pth')
    print()
    print('NEXT STEPS:')
    print('  1. Go to Output tab -> New Dataset')
    print('  2. Upload all .pth and .pkl files')
    print('  3. Set INFERENCE_ONLY = True')
    print('  4. Update MODEL_PATH to your dataset path')
    print('  5. Run again -> submission.csv in ~40 min')
    print('='*50)